<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/YapaySinirAglariFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sayın Hocam,

Ödevi incelerken aşağıdaki hususları dikkate almanızı rica ederim.

Kaggle veri indirme ve yükleme süreçlerinde, veri boyutunun yüksek olması nedeniyle doğrudan klasör olarak indirme yapılamamıştır. Bu nedenle veri, Kaggle API kullanılarak .json formatında indirilmiş ve Colab ortamına yüklenmiştir. Colab üzerinde veri erişimi için ilgili kaggle.json dosyasının yolu tanımlanarak API yetkilendirmesi gerçekleştirilmiştir. Uygulamanın çalıştırılabilmesi için önceden KAGGLE_USERNAME ve KAGGLE_KEY değişkenlerinin tanımlanması gerekmektedir.



Bilgilerinize sunarım.

Saygılarımla

Soru 1- Veri Seti ve Sınıf Seçimi

In [ ]:

from google.colab import userdata
import os
import zipfile

In [ ]:
from google.colab import userdata
import os

print("Colab Secrets'tan Kaggle API bilgileri okunuyor...")
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("API bilgileri başarıyla okundu.")
except userdata.SecretNotFoundError:
    print("\nHata: 'KAGGLE_USERNAME' veya 'KAGGLE_KEY' gizli anahtarları bulunamadı.")
    print("Lütfen yukarıdaki adımları takip ederek anahtarları doğru şekilde eklediğinizden emin olun.")
except Exception as e:
    print(f"\nBeklenmeyen bir hata oluştu: {e}")


In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets download -d nodoubttome/skin-cancer9-classesisic --force

In [ ]:
zip_file_path = '/content/skin-cancer9-classesisic.zip'
extract_dir = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")

print("Çıkarılan dizinin içeriği:")
for item in os.listdir(extract_dir):
    print(item)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from PIL import Image
import seaborn as sns
import tensorflow as tf


import keras
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import image

from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split

### Resimlerin Yüklenmesi ve Ön İşleme


In [ ]:


# Veri setinin ana dizin yolu
main_data_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
print(f"Ana veri dizini: {main_data_dir}\n")

# Ana dizinin içeriği
print("Ana Dizin İçeriği:")
main_dir_contents = os.listdir(main_data_dir)
for item in main_dir_contents:
    print(f"- {item}")

print("\nAlt Dizinlerin İçeriği:")
# Ana dizin içindeki her bir alt dizinin içeriğini listele
for item in main_dir_contents:
    item_path = os.path.join(main_data_dir, item)
    if os.path.isdir(item_path):
        print(f"\n--- {item} İçeriği ---")
        sub_dir_contents = os.listdir(item_path)
        for sub_item in sub_dir_contents:
            print(f"  - {sub_item}")
    elif os.path.isfile(item_path):
        print(f"\n--- {item} (Dosya) ---")


In [ ]:

main_data_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
total_images = 0
image_extensions = ['.jpg', '.jpeg', '.png']
print(f"'{main_data_dir}' dizini taranıyor...")
for dirpath, dirnames, filenames in os.walk(main_data_dir):
    for filename in filenames:
        if any(filename.lower().endswith(ext) for ext in image_extensions):
            total_images += 1

print(f"Veri setindeki toplam görüntü sayısı: {total_images}")

In [ ]:



class_counts = {}

main_data_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
sub_dirs_to_scan = ['Train', 'Test']

image_extensions = ['.jpg', '.jpeg', '.png']
print("Sınıf bazında görüntü sayıları hesaplanıyor...")
for sub_dir_name in sub_dirs_to_scan:
    current_path = os.path.join(main_data_dir, sub_dir_name)
    for dirpath, dirnames, filenames in os.walk(current_path):
        if filenames and any(f.lower().endswith(ext) for f in filenames for ext in image_extensions):
            class_name = os.path.basename(dirpath)
            count = 0
            for filename in filenames:
                if any(filename.lower().endswith(ext) for ext in image_extensions):
                    count += 1
            if class_name in class_counts:
                class_counts[class_name] += count
            else:
                class_counts[class_name] = count

class_distribution = pd.Series(class_counts)

print("\nHer sınıf için hesaplanan resim sayıları:")
print(class_distribution)
plt.figure(figsize=(12, 6))
sns.barplot(x=class_distribution.index, y=class_distribution.values, palette='viridis')
plt.title('Veri Seti Sınıf Dağılımı', fontsize=16)
plt.xlabel('Sınıflar', fontsize=12)
plt.ylabel('Resim Sayısı', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
sorted_classes = class_distribution.sort_values(ascending=False)
top_2_classes = sorted_classes.head(2)

print("En çok görüntüye sahip iki sınıf:")
print(top_2_classes)

In [ ]:
selected_classes = top_2_classes.index.tolist()
print(f"İkili sınıflandırma için seçilen sınıflar: {selected_classes}")

binary_image_paths = []
binary_image_labels = []

main_data_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
image_extensions = ['.jpg', '.jpeg', '.png']

print("Belirlenen sınıflara ait görüntü yolları ve etiketler toplanıyor...")

for sub_dir_name in ['Train', 'Test']:
    current_path = os.path.join(main_data_dir, sub_dir_name)
    for dirpath, dirnames, filenames in os.walk(current_path):
        class_name = os.path.basename(dirpath)
        if class_name in selected_classes:
            for filename in filenames:
                if any(filename.lower().endswith(ext) for ext in image_extensions):
                    full_path = os.path.join(dirpath, filename)
                    binary_image_paths.append(full_path)

                    # Assign binary labels: 0 for the first class, 1 for the second
                    if class_name == selected_classes[0]:
                        binary_image_labels.append(0)
                    elif class_name == selected_classes[1]:
                        binary_image_labels.append(1)

binary_df = pd.DataFrame({
    'Path': binary_image_paths,
    'Label': binary_image_labels
})

print("\nOluşturulan ikili DataFrame'in ilk birkaç satırı:")
print(binary_df.head())

print("\nDataFrame bilgisi:")
binary_df.info()

print("\nEtiket dağılımı:")
print(binary_df['Label'].value_counts())


In [ ]:
print(f"İkili sınıflandırma için seçilen sınıflar: {selected_classes[0]} ve {selected_classes[1]}\n")

class_0_name = selected_classes[0]
class_1_name = selected_classes[1]

class_counts_binary = binary_df['Label'].value_counts().sort_index()

print(f"{class_0_name} (0) için görüntü sayısı: {class_counts_binary.get(0, 0)}")
print(f"{class_1_name} (1) için görüntü sayısı: {class_counts_binary.get(1, 0)}")

total_images_binary = binary_df.shape[0]
print(f"İkili sınıflandırma için toplam görüntü sayısı: {total_images_binary}")
count_0 = class_counts_binary.get(0, 0)
count_1 = class_counts_binary.get(1, 0)

if count_0 != count_1:
    difference = abs(count_0 - count_1)
    print(f"\nSınıflar arasında bir dengesizlik bulunmaktadır. '{class_0_name}' sınıfında {count_0} görüntü, '{class_1_name}' sınıfında {count_1} görüntü vardır. Fark: {difference} görüntü.")
else:
    print("\nSınıflar arasında denge bulunmaktadır.")

In [ ]:
from sklearn.utils import class_weight
import numpy as np

print("Sınıf ağırlıkları hesaplanıyor...")


class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Hesaplanan ağırlıkları bir sözlüğe dönüştürme, böylece model eğitiminde kolayca kullanılabilir.
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}

print("Sınıf ağırlıkları başarıyla hesaplandı.")
print("Hesaplanan sınıf ağırlıkları:")
print(class_weights_dict)


In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(x=class_counts_binary.index, y=class_counts_binary.values, palette='viridis')
plt.title('İkili Sınıf Dağılımı', fontsize=16)
plt.xlabel('Sınıflar (0: ' + class_0_name + ', 1: ' + class_1_name + ')', fontsize=12)
plt.ylabel('Resim Sayısı', fontsize=12)
plt.xticks(ticks=[0, 1], labels=[class_0_name, class_1_name], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:

# Hedef görüntü boyutu
IMG_HEIGHT = 224
IMG_WIDTH = 224
TARGET_SIZE = (IMG_HEIGHT, IMG_WIDTH)

print(f"Hedef görüntü boyutu: {TARGET_SIZE}")
print(f"binary_df içinde {len(binary_df)} görüntü yolu bulunmaktadır.")
X_data_list = []
y_data_list = []

# binary_df DataFrame'ini döngüye alarak her bir görüntü yolu ve etiketi için işlemleri
print("Görüntüler yükleniyor, yeniden boyutlandırılıyor ve normalize ediliyor...")
for index, row in binary_df.iterrows():
    img_path = row['Path']
    label = row['Label']

    try:

        img = image.load_img(img_path, target_size=TARGET_SIZE)
        img_array = image.img_to_array(img)
        img_array = img_array / 255.0

        X_data_list.append(img_array)
        y_data_list.append(label)
    except Exception as e:
        print(f"Hata oluştu '{img_path}': {e}")

# Görüntü ve etiket listelerini NumPy dizilerine dönüştürme
X_data = np.array(X_data_list)
y_data = np.array(y_data_list)

print(f"\nToplam yüklenen ve ön işlenmiş görüntü sayısı: {len(X_data_list)}")
print(f"X_data şekli: {X_data.shape}")
print(f"y_data şekli: {y_data.shape}")

In [ ]:

num_images_to_display = 5

plt.figure(figsize=(15, 5))
for i in range(num_images_to_display):
    plt.subplot(1, num_images_to_display, i + 1)
    plt.imshow(X_data[i])

    original_label_name = selected_classes[y_data[i]]
    plt.title(f"Label: {original_label_name}")
    plt.axis('off')
plt.suptitle(f"Ön İşlenmiş Görüntüler (Boyut: {IMG_HEIGHT}x{IMG_WIDTH}, Normalize Edildi)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print(f"İlk {num_images_to_display} ön işlenmiş görüntü görselleştirildi.")

Soru 2- Veri Bölme: Train / Validation / Test

In [ ]:
print("Veri setini eğitim, doğrulama ve test setlerine bölme işlemi başlatılıyor...")

# İlk bölme: %70 eğitim, %30 geçici (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.3, stratify=y_data, random_state=42)

# İkinci bölme: Geçici setleri %15 doğrulama, %15 test olarak ayırma
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"\nX_train şekli: {X_train.shape}, y_train şekli: {y_train.shape}")
print(f"X_val şekli: {X_val.shape}, y_val şekli: {y_val.shape}")
print(f"X_test şekli: {X_test.shape}, y_test şekli: {y_test.shape}")

print("Veri setleri başarıyla bölündü.")

In [ ]:
print("\nEğitim, Doğrulama ve Test Setlerindeki Görüntü Sayıları ve Sınıf Dağılımları:\n")

# Eğitim Seti
print(f"Eğitim Seti (X_train, y_train):")
print(f"  Toplam Görüntü Sayısı: {len(y_train)}")
print("  Sınıf Dağılımı:")
print(pd.Series(y_train).value_counts().sort_index().rename({0: class_0_name, 1: class_1_name}))

# Doğrulama Seti
print(f"\nDoğrulama Seti (X_val, y_val):")
print(f"  Toplam Görüntü Sayısı: {len(y_val)}")
print("  Sınıf Dağılımı:")
print(pd.Series(y_val).value_counts().sort_index().rename({0: class_0_name, 1: class_1_name}))

# Test Seti
print(f"\nTest Seti (X_test, y_test):")
print(f"  Toplam Görüntü Sayısı: {len(y_test)}")
print("  Sınıf Dağılımı:")
print(pd.Series(y_test).value_counts().sort_index().rename({0: class_0_name, 1: class_1_name}))

In [ ]:


random_seed = 42

print("NumPy dizilerinden tf.data.Dataset nesneleri oluşturuluyor...")

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

print(f"Eğitim veri seti oluşturuldu: {train_ds.element_spec}")
print(f"Doğrulama veri seti oluşturuldu: {val_ds.element_spec}")
print(f"Test veri seti oluşturuldu: {test_ds.element_spec}")

In [ ]:
print("Veri artırma modeli güncelleniyor...")

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=random_seed),
    tf.keras.layers.RandomRotation(factor=0.065, seed=random_seed),
    tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1, seed=random_seed),
    tf.keras.layers.RandomBrightness(factor=0.1, seed=random_seed),
    tf.keras.layers.RandomContrast(factor=0.1, seed=random_seed)
], name='data_augmentation')

print("Veri artırma modeli başarıyla güncellendi.")

# BATCH_SIZE değişkenini tanımlama
BATCH_SIZE = 32
print(f"BATCH_SIZE olarak {BATCH_SIZE} ayarlandı.")

print("Veri setlerine veri artırma ve performans iyileştirmeleri uygulanıyor...")

# Veri artırmayı eğitim veri setine uygulama
def apply_data_augmentation(image, label):
    image = data_augmentation(image)
    return image, label

train_ds = train_ds.map(apply_data_augmentation, num_parallel_calls=tf.data.AUTOTUNE)

# Performans iyileştirmeleri
train_ds = train_ds.shuffle(buffer_size=1000, seed=random_seed).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

print("Veri setlerine veri artırma ve performans iyileştirmeleri başarıyla uygulandı.")

In [ ]:
print('Eğitim veri setinden artırılmış görüntü örnekleri görselleştiriliyor...')

# Eğitim veri setinden bir parti al
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        original_label_name = selected_classes[labels[i].numpy()]
        plt.title(f"Label: {original_label_name}")
        plt.axis("off")
    plt.suptitle('Artırılmış Eğitim Verisi Örnekleri', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print('Artırılmış görüntü örnekleri başarıyla görselleştirildi.')

Soru 3- Input Pipeline ve Data Augmentation

In [ ]:
print("Uygulanan Veri Artırma Teknikleri ve Parametreleri:")
for layer in data_augmentation.layers:
    print(f"- Teknik: {layer.name}")
    config = layer.get_config()
    if 'factor' in config:
        print(f"  Parametre (factor): {config['factor']}")
    if 'height_factor' in config and 'width_factor' in config:
        print(f"  Parametre (zoom): height_factor={config['height_factor']}, width_factor={config['width_factor']}")
    if 'mode' in config:
        print(f"  Parametre (mode): {config['mode']}")
    if 'rotation_factor' in config:
        print(f"  Parametre (rotation_factor): {config['rotation_factor']}")
    if 'fill_mode' in config:
        print(f"  Parametre (fill_mode): {config['fill_mode']}")
    if 'interpolation' in config:
        print(f"  Parametre (interpolation): {config['interpolation']}")
    if 'seed' in config:
        print(f"  Parametre (seed): {config['seed']}")
    print("\n")

In [ ]:
print('Eğitim veri setinden artırılmış görüntü örnekleri görselleştiriliyor...')

# Eğitim veri setinden bir parti al
for images, labels in train_ds.take(1):
    num_images_to_display = min(len(images), 6)
    plt.figure(figsize=(15, 5))
    for i in range(num_images_to_display):
        ax = plt.subplot(1, num_images_to_display, i + 1)

        # Görüntü verisini [0, 1] aralığına klipleyerek uyarıları önle
        plt.imshow(np.clip(images[i].numpy(), 0.0, 1.0))

        original_label_name = selected_classes[labels[i].numpy()]
        plt.title(f"Label: {original_label_name}")
        plt.axis("off")
    plt.suptitle('Uygulanan Veri Artırma Sonrası Eğitim Verisi Örnekleri', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print('Artırılmış görüntü örnekleri başarıyla görselleştirildi.')

Soru 4- Model-1: Scratch CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

print('Scratch CNN modeli oluşturuluyor...')

def create_scratch_cnn_model(input_shape):
    model = tf.keras.Sequential([
        # 1. Giriş katmanı (Hata almamak için açıkça tanımlı)
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # 2. Grad-CAM için Kritik Adım: Son Evrişimsel Katman
        # 'name' parametresi ile isimlendiriyoruz ki Grad-CAM burayı hedef alabilsin.
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', name='final_conv_layer'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(1, activation='sigmoid') # İkili sınıflandırma
    ], name='Scratch_CNN_Model')

    return model

# Modeli oluşturma
# Not: IMG_HEIGHT ve IMG_WIDTH değerlerinizin 224 olduğunu varsayıyorum
input_shape = (224, 224, 3)
scratch_cnn_model = create_scratch_cnn_model(input_shape)

# --- KRİTİK ADIM: MODELİ İNŞA ETME (BUILD) ---
# AttributeError hatasını önlemek için modele boş bir veri (dummy data)
# göndererek 'never been called' durumundan çıkarıyoruz.
dummy_data = np.zeros((1, *input_shape))
_ = scratch_cnn_model(dummy_data)

print('Scratch CNN modeli başarıyla oluşturuldu ve analiz için hazırlandı.')
scratch_cnn_model.summary()

In [ ]:
print('Scratch CNN modeli oluşturuluyor...')

def create_scratch_cnn_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=input_shape), # Addressing the UserWarning explicitly
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(1, activation='sigmoid') # İkili sınıflandırma için
    ], name='Scratch_CNN_Model')
    return model

# Modeli oluşturma
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
scratch_cnn_model = create_scratch_cnn_model(input_shape)

print('Scratch CNN modeli başarıyla oluşturuldu.')
scratch_cnn_model.summary()

# Modelin derlenmesi
LEARNING_RATE = 0.001
print(f"\nModel derleniyor (Learning Rate: {LEARNING_RATE})...")
scratch_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
print("Model başarıyla derlendi.")

# Callback'lerin oluşturulması
print("\nCallback'ler oluşturuluyor...")
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)

callbacks = [early_stopping, reduce_lr]
print("Callback'ler başarıyla oluşturuldu: EarlyStopping, ReduceLROnPlateau.")

# Modeli eğitme
EPOCHS = 100
print(f"\nModel {EPOCHS} epoch boyunca eğitiliyor...")
history = scratch_cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)
print("Model eğitimi tamamlandı.")

# Eğitim geçmişinin görselleştirilmesi
print("\nEğitim geçmişi görselleştiriliyor...")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('Model Doğruluk Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Model Kayıp Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
print("Eğitim geçmişi başarıyla görselleştirildi.")

# Modelin performans analizi ve özeti
print("\nModelin performans analizi ve özeti:")
print("--------------------------------------")

best_val_accuracy = max(history.history['val_accuracy'])
best_val_loss = min(history.history['val_loss'])
print(f"En iyi Doğrulama Doğruluğu: {best_val_accuracy:.4f}")
print(f"En düşük Doğrulama Kaybı: {best_val_loss:.4f}")

# Overfitting/Underfitting analizi
final_train_accuracy = history.history['accuracy'][-1]
final_val_accuracy = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"Son Eğitim Doğruluğu: {final_train_accuracy:.4f}")
print(f"Son Doğrulama Doğruluğu: {final_val_accuracy:.4f}")
print(f"Son Eğitim Kaybı: {final_train_loss:.4f}")
print(f"Son Doğrulama Kaybı: {final_val_loss:.4f}")

if final_train_accuracy > final_val_accuracy and abs(final_train_accuracy - final_val_accuracy) > 0.1:
    print("Modelde overfitting (aşırı uyum) belirtileri gözlemleniyor. Eğitim doğruluğu doğrulama doğruluğundan önemli ölçüde yüksek.")
elif final_val_accuracy > final_train_accuracy:
    print("Modelde underfitting (eksik uyum) belirtileri olabilir veya daha fazla eğitim gerekebilir.")
else:
    print("Model eğitim ve doğrulama setlerinde dengeli bir performans sergiliyor.")

print("\nScratch CNN Modeli Özeti:")
print("------------------------")
print("Mimari:")
print("  - Üç adet Conv2D katmanı (32, 64, 128 filtre) ve her birinin ardından Batch Normalization ve MaxPooling (GlobalAveragePooling son Conv katmanından sonra).")
print("  - Bir Dense katmanı (128 ünite), ardından Batch Normalization ve Dropout (0.5).")
print("  - Son çıktı katmanı: 1 üniteli Dense katmanı, sigmoid aktivasyonu (ikili sınıflandırma).")
print("Eğitim Ayarları:")
print(f"  - Optimizasyon: Adam (Öğrenme Oranı: {LEARNING_RATE})")
print("  - Kayıp Fonksiyonu: BinaryCrossentropy")
print("  - Metrikler: Accuracy")
print("  - Batch Boyutu: 32")
print(f"  - Epoch Sayısı: {EPOCHS} (Ancak EarlyStopping ile daha erken durdurulabilir)")
print("Callback'ler:")
print("  - EarlyStopping: val_loss monitör edilir, 10 patience, en iyi ağırlıklar geri yüklenir.")
print("  - ReduceLROnPlateau: val_loss monitör edilir, factor 0.2, 5 patience, minimum öğrenme oranı 1e-6.")
print("Performans Analizi:")
print("  - Eğri çizimleri, modelin eğitim ve doğrulama performansı arasındaki ilişkiyi göstermektedir.")
print("  - Kayıp eğrisinin azalması ve doğruluk eğrisinin artması, modelin öğrendiğini gösterir.")
print("  - Eğitim ve doğrulama eğrileri arasındaki yakınlık, modelin genelleme yeteneğini gösterir.")

Soru 5- Model-2 ve Model-3: Transfer Learning

In [ ]:
print("MobileNetV2 feature extraction modeli oluşturuluyor...")

# 1. MobileNetV2 modelini 'imagenet' ağırlıklarıyla, include_top=False ve input_shape parametreleriyle yükleyin.
base_model_mobilenetv2 = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)

# 2. base_model_mobilenetv2 modelinin tüm katmanlarını dondurun (freeze).
base_model_mobilenetv2.trainable = False

# 3. Modelin üst kısmına (classification head) aşağıdaki katmanları ekleyerek yeni bir model oluşturun:
inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model_mobilenetv2(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x) # 0.3-0.5 arası bir oran
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

mobilenetv2_feature_extractor = tf.keras.Model(inputs, outputs)

print("MobileNetV2 feature extraction modeli başarıyla oluşturuldu.")

# 5. Modelin özetini (summary()) yazdırın.
mobilenetv2_feature_extractor.summary()

In [ ]:
print("MobileNetV2 feature extraction modeli derleniyor ve eğitiliyor...")

# Modelin derlenmesi
LEARNING_RATE_FE = 0.001
print(f"\nModel derleniyor (Learning Rate: {LEARNING_RATE_FE})...")
mobilenetv2_feature_extractor.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_FE),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
print("Model başarıyla derlendi.")

# Callback'lerin oluşturulması
print("\nCallback'ler oluşturuluyor...")
early_stopping_fe = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr_fe = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)

callbacks_fe = [early_stopping_fe, reduce_lr_fe]
print("Callback'ler başarıyla oluşturuldu: EarlyStopping, ReduceLROnPlateau.")

# Modeli eğitme
EPOCHS_FE = 100
print(f"\nModel {EPOCHS_FE} epoch boyunca eğitiliyor...")
history_mobilenetv2_fe = mobilenetv2_feature_extractor.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FE,
    callbacks=callbacks_fe
)
print("Model eğitimi tamamlandı.")

# Eğitim geçmişinin görselleştirilmesi
print("\nEğitim geçmişi görselleştiriliyor...")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_mobilenetv2_fe.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history_mobilenetv2_fe.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('MobileNetV2 Feature Extraction Model Doğruluk Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_mobilenetv2_fe.history['loss'], label='Eğitim Kaybı')
plt.plot(history_mobilenetv2_fe.history['val_loss'], label='Doğrulama Kaybı')
plt.title('MobileNetV2 Feature Extraction Model Kayıp Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
print("Eğitim geçmişi başarıyla görselleştirildi.")

# Modelin performans analizi ve özeti
print("\nMobileNetV2 Feature Extraction Modelinin Performans Analizi ve Özeti:")
print("----------------------------------------------------------------")

best_val_accuracy_fe = max(history_mobilenetv2_fe.history['val_accuracy'])
best_val_loss_fe = min(history_mobilenetv2_fe.history['val_loss'])
print(f"En iyi Doğrulama Doğruluğu: {best_val_accuracy_fe:.4f}")
print(f"En düşük Doğrulama Kaybı: {best_val_loss_fe:.4f}")

final_train_accuracy_fe = history_mobilenetv2_fe.history['accuracy'][-1]
final_val_accuracy_fe = history_mobilenetv2_fe.history['val_accuracy'][-1]
final_train_loss_fe = history_mobilenetv2_fe.history['loss'][-1]
final_val_loss_fe = history_mobilenetv2_fe.history['val_loss'][-1]

print(f"Son Eğitim Doğruluğu: {final_train_accuracy_fe:.4f}")
print(f"Son Doğrulama Doğruluğu: {final_val_accuracy_fe:.4f}")
print(f"Son Eğitim Kaybı: {final_train_loss_fe:.4f}")
print(f"Son Doğrulama Kaybı: {final_val_loss_fe:.4f}")

if final_train_accuracy_fe > final_val_accuracy_fe and abs(final_train_accuracy_fe - final_val_accuracy_fe) > 0.1:
    print("Modelde overfitting (aşırı uyum) belirtileri gözlemleniyor. Eğitim doğruluğu doğrulama doğruluğundan önemli ölçüde yüksek.")
elif final_val_accuracy_fe > final_train_accuracy_fe:
    print("Modelde underfitting (eksik uyum) belirtileri olabilir veya daha fazla eğitim gerekebilir.")
else:
    print("Model eğitim ve doğrulama setlerinde dengeli bir performans sergiliyor.")


In [ ]:
print("MobileNetV2 base model katmanlarını çözme işlemi başlatılıyor...")

num_layers = len(base_model_mobilenetv2.layers)
print(f"MobileNetV2 base modelinde toplam {num_layers} katman bulunmaktadır.")

num_unfreeze = int(num_layers * 0.25)

for layer in base_model_mobilenetv2.layers:
    layer.trainable = False

print(f"Son {num_unfreeze} katman eğitilebilir olarak ayarlanacak.")

for layer in base_model_mobilenetv2.layers[num_layers - num_unfreeze:]:
    layer.trainable = True

print("MobileNetV2 base model katmanlarının trainable durumu güncellendi.")


print("\nMobileNetV2 Base Model Katmanlarının Trainable Durumu:")
for i, layer in enumerate(base_model_mobilenetv2.layers):
    if i >= num_layers - num_unfreeze:
        print(f"  Katman {i}: {layer.name} - Trainable: {layer.trainable}")


In [ ]:
print("MobileNetV2 fine-tuning modeli derleniyor ve eĞitiliyor...")

LEARNING_RATE_FT = 0.0001
print(f"\nModel yeniden derleniyor (Learning Rate: {LEARNING_RATE_FT})...")
mobilenetv2_feature_extractor.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_FT),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
print("Model başırıyla yeniden derlendi.")

# 4. Yeniden derlenmiş mobilenetv2_feature_extractor modelini eğitin.
EPOCHS_FT = 100
print(f"\nModel {EPOCHS_FT} epoch boyunca fine-tuning ile eğitiliyor...")
history_mobilenetv2_ft = mobilenetv2_feature_extractor.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks_fe # Daha önce tanımlanan callback'leri kullanıyoruz
)
print("Fine-tuning eğitimi tamamlandı.")

# 5. Modelin fine-tuning eğitim geçmişini görsellenmesini sağlayın.
print("\nFine-tuning eğitim geçmişinin görsellenmesi...")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_mobilenetv2_ft.history['accuracy'], label='Eğitim Doğurluğunu (Fine-tuning)')
plt.plot(history_mobilenetv2_ft.history['val_accuracy'], label='Doğurlama Doğurluğunu (Fine-tuning)')
plt.title('MobileNetV2 Fine-tuning Model Doğurluk Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Doğurluk')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_mobilenetv2_ft.history['loss'], label='Eğitim Kaybı (Fine-tuning)')
plt.plot(history_mobilenetv2_ft.history['val_loss'], label='Doğurlama Kaybı (Fine-tuning)')
plt.title('MobileNetV2 Fine-tuning Model Kayıp Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
print("Fine-tuning eğitim geçmişinin görsellenmesi tamamlandı.")

# Modelin performans analizi ve özeti
print("\nMobileNetV2 Fine-tuning Modelinin Performans Analizi ve Özeti:")
print("----------------------------------------------------------------")

best_val_accuracy_ft = max(history_mobilenetv2_ft.history['val_accuracy'])
best_val_loss_ft = min(history_mobilenetv2_ft.history['val_loss'])
print(f"En iyi Doğrulama Doğurluğunu (Fine-tuning): {best_val_accuracy_ft:.4f}")
print(f"En düşük Doğrulama Kaybı (Fine-tuning): {best_val_loss_ft:.4f}")

final_train_accuracy_ft = history_mobilenetv2_ft.history['accuracy'][-1]
final_val_accuracy_ft = history_mobilenetv2_ft.history['val_accuracy'][-1]
final_train_loss_ft = history_mobilenetv2_ft.history['loss'][-1]
final_val_loss_ft = history_mobilenetv2_ft.history['val_loss'][-1]

print(f"Son Eğitim Doğurluğunu (Fine-tuning): {final_train_accuracy_ft:.4f}")
print(f"Son Doğurlama Doğurluğunu (Fine-tuning): {final_val_accuracy_ft:.4f}")
print(f"Son Eğitim Kaybı (Fine-tuning): {final_train_loss_ft:.4f}")
print(f"Son Doğurlama Kaybı (Fine-tuning): {final_val_loss_ft:.4f}")

if final_train_accuracy_ft > final_val_accuracy_ft and abs(final_train_accuracy_ft - final_val_accuracy_ft) > 0.1:
    print("Modelde overfitting (aşırı uyum) belirtileri gözlemleniyor. Eğitim doğurluğunu doğurlama doğurluğundan önemli ölçüde yüksek.")
elif final_val_accuracy_ft > final_train_accuracy_ft:
    print("Modelde underfitting (eksik uyum) belirtileri olabilir veya daha fazla eğitim gerekebilir.")
else:
    print("Model eğitim ve doğurlama setlerinde dengeli bir performans sergiliyor.")

print("\n--- Feature Extraction ve Fine-tuning Karşılaşturması ---")
print(f"Feature Extraction En iyi Doğrulama Doğurluğunu: {best_val_accuracy_fe:.4f}")
print(f"Fine-tuning En iyi Doğrulama Doğurluğunu: {best_val_accuracy_ft:.4f}")
print(f"Feature Extraction En düşük Doğrulama Kaybı: {best_val_loss_fe:.4f}")
print(f"Fine-tuning En düşük Doğrulama Kaybı: {best_val_loss_ft:.4f}")

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

print("EfficientNetB0 feature extraction modeli oluşturuluyor...")


base_model_efficientnetb0 = tf.keras.applications.EfficientNetB0(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)

base_model_efficientnetb0.trainable = False


inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model_efficientnetb0(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

efficientnetb0_feature_extractor = tf.keras.Model(inputs, outputs)

print("EfficientNetB0 feature extraction modeli başarıyla oluşturuldu.")


efficientnetb0_feature_extractor.summary()

In [ ]:
print("EfficientNetB0 fine-tuning işlemleri baŒlatılıyor...")


num_layers_efficientnetb0 = len(base_model_efficientnetb0.layers)

num_unfreeze_efficientnetb0 = int(num_layers_efficientnetb0 * 0.25)

for layer in base_model_efficientnetb0.layers:
    layer.trainable = False


for layer in base_model_efficientnetb0.layers[num_layers_efficientnetb0 - num_unfreeze_efficientnetb0:]:
    layer.trainable = True

print(f"EfficientNetB0 base modelinde toplam {num_layers_efficientnetb0} katman bulunmaktadır.")
print(f"Son {num_unfreeze_efficientnetb0} katman eĒitilebilir olarak ayarlandı.")

print("\nEfficientNetB0 Base Model Katmanlarının Trainable Durumu:")
for i, layer in enumerate(base_model_efficientnetb0.layers):
    if i >= num_layers_efficientnetb0 - num_unfreeze_efficientnetb0:
        print(f"  Katman {i}: {layer.name} - Trainable: {layer.trainable}")

# EfficientNetB0 Feature Extraction için callbacks'leri tanımlayın
early_stopping_en_fe = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr_en_fe = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)

LEARNING_RATE_EN_FT = 0.0001
print(f"\nModel yeniden derleniyor (Learning Rate: {LEARNING_RATE_EN_FT})...")
efficientnetb0_feature_extractor.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_EN_FT),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)
print("Model baş-arıyla yeniden derlendi.")


EPOCHS_FT = 100
print(f"\nModel {EPOCHS_FT} epoch boyunca fine-tuning ile eĒitiliyor...")
history_efficientnetb0_ft = efficientnetb0_feature_extractor.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=[early_stopping_en_fe, reduce_lr_en_fe]
)
print("Fine-tuning eĒitimi tamamlandı.")


print("\nFine-tuning eğitim geçmişinin görselleştirilmesi...")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_efficientnetb0_ft.history['accuracy'], label='Eğitim Doğ-ruluğ-u (Fine-tuning)')
plt.plot(history_efficientnetb0_ft.history['val_accuracy'], label='Doğ-rulama Doğ-ruluğ-u (Fine-tuning)')
plt.title('EfficientNetB0 Fine-tuning Model Doğ-ruluk Eğ-risi')
plt.xlabel('Epoch')
plt.ylabel('Doğ-ruluk')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_efficientnetb0_ft.history['loss'], label='Eğitim Kaybı (Fine-tuning)')
plt.plot(history_efficientnetb0_ft.history['val_loss'], label='Doğ-rulama Kaybı (Fine-tuning)')
plt.title('EfficientNetB0 Fine-tuning Model Kayıp Eğrisi')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
print("Fine-tuning eğitim geçmişinin görselleştirilmesi tamamlandı.")

print("\nEfficientNetB0 Fine-tuning Modelinin Performans Analizi ve Ozeti:")
print("----------------------------------------------------------------")

best_val_accuracy_en_ft = max(history_efficientnetb0_ft.history['val_accuracy'])
best_val_loss_en_ft = min(history_efficientnetb0_ft.history['val_loss'])
print(f"En iyi Dogrulama (Fine-tuning): {best_val_accuracy_en_ft:.4f}")
print(f"En dusuk Dogrulama Kaybı (Fine-tuning): {best_val_loss_en_ft:.4f}")

final_train_accuracy_en_ft = history_efficientnetb0_ft.history['accuracy'][-1]
final_val_accuracy_en_ft = history_efficientnetb0_ft.history['val_accuracy'][-1]
final_train_loss_en_ft = history_efficientnetb0_ft.history['loss'][-1]
final_val_loss_en_ft = history_efficientnetb0_ft.history['loss'][-1]

print(f"Son Eğitim Dogrulugu (Fine-tuning): {final_train_accuracy_en_ft:.4f}")
print(f"Son Doğ-rulama Dogrulugu (Fine-tuning): {final_val_accuracy_en_ft:.4f}")
print(f"Son Eğitim Kaybı (Fine-tuning): {final_train_loss_en_ft:.4f}")
print(f"Son Dogrulama Kaybı (Fine-tuning): {final_val_loss_en_ft:.4f}")

if final_train_accuracy_en_ft > final_val_accuracy_en_ft and abs(final_train_accuracy_en_ft - final_val_accuracy_en_ft) > 0.1:
    print("Modelde overfitting belirtileri gozlemleniyor. ")
elif final_val_accuracy_en_ft > final_train_accuracy_en_ft:
    print("Modelde underfitting (eksik uyum) belirtileri olabilir veya daha fazla eğitim gerekebilir.")
else:
    print("Model eğitim ve doğ-rulama setlerinde dengeli bir performans sergiliyor.")

# Feature Extraction'ın en iyi değerlerini tanımlayın (önceki hücreden)
best_val_accuracy_en_fe = 0.5143 # Bu değer, önceki hücredeki EfficientNetB0 Feature Extraction sonucuna göre güncellenmelidir.
best_val_loss_en_fe = 0.6927 # Bu değer de önceki hücredeki EfficientNetB0 Feature Extraction sonucuna göre güncellenmelidir.

print("\n--- Feature Extraction ve Fine-tuning Karş-ılaş-tırması ---")
print(f"Feature Extraction En iyi Dogrulama Doğ-ruluğ-u: {best_val_accuracy_en_fe:.4f}")
print(f"Fine-tuning En iyi Dogrulama Doğ-ruluğ-u: {best_val_accuracy_en_ft:.4f}")
print(f"Feature Extraction En dusuk Dogrulama Kaybı: {best_val_loss_en_fe:.4f}")
print(f"Fine-tuning En dusuk Dogrulama Kaybı: {best_val_loss_en_ft:.4f}")

In [ ]:
print("EfficientNetB0 fine-tuning modelinin test veri seti üzerinde değerlendirilmesi...")


loss_en_ft, accuracy_en_ft = efficientnetb0_feature_extractor.evaluate(test_ds)

print(f"EfficientNetB0 Fine-tuning Test Kaybı: {loss_en_ft:.4f}")
print(f"EfficientNetB0 Fine-tuning Test Doğruluğu: {accuracy_en_ft:.4f}")

print("\n--- Tüm Modellerin Performans Özeti ---")


best_val_accuracy = 0.5143
best_val_loss = 0.6896

best_val_accuracy_fe = 0.6286
best_val_loss_fe = 0.6872

best_val_accuracy_en_fe = 0.5143
best_val_loss_en_fe = 0.6927


best_val_accuracy_en_ft = 0.5143
best_val_loss_en_ft = 0.6926

print(f"\nScratch CNN (Validation): Doğruluk = {best_val_accuracy:.4f}, Kayıp = {best_val_loss:.4f}")
print(f"MobileNetV2 Feature Extraction (Validation): Doğruluk = {best_val_accuracy_fe:.4f}, Kayıp = {best_val_loss_fe:.4f}")
print(f"EfficientNetB0 Feature Extraction (Validation): Doğruluk = {best_val_accuracy_en_fe:.4f}, Kayıp = {best_val_loss_en_fe:.4f}")
print(f"EfficientNetB0 Fine-tuning (Test): Doğruluk = {accuracy_en_ft:.4f}, Kayıp = {loss_en_ft:.4f}")

print("\n--- Modellerin Karşılaştırmalı Performansı ---")
print("1. Scratch CNN Model: Bu model, en düşük doğrulama doğruluğuna ve en yüksek kayba sahiptir. Kendi başına öğrenmeye çalıştığı için performansının daha düşük olması beklenir.")
print(f"   - En iyi Doğrulama Doğruluğu: {best_val_accuracy:.4f}")
print(f"   - En düşük Doğrulama Kaybı: {best_val_loss:.4f}")
print("\n2. MobileNetV2 Feature Extraction Model: Bu model, en iyi doğrulama doğruluğu açısından en yüksek performansı göstermiştir. Önceden eğitilmiş ağırlıkları dondurarak ve sadece sınıflandırma katmanını eğiterek daha iyi genelleme yeteneği kazanmıştır.")
print(f"   - En iyi Doğrulama Doğruluğu: {best_val_accuracy_fe:.4f}")
print(f"   - En düşük Doğrulama Kaybı: {best_val_loss_fe:.4f}")
print("\n3. EfficientNetB0 Feature Extraction Model: Bu modelin doğrulama doğruluğu Scratch CNN modeline yakın olup, MobileNetV2'den daha düşük kalmıştır. Ancak test veri seti üzerinde benzer bir doğruluk sergilemiştir.")
print(f"   - En iyi Doğrulama Doğruluğu: {best_val_accuracy_en_fe:.4f}")
print(f"   - En düşük Doğrulama Kaybı: {best_val_loss_en_fe:.4f}")
print("\n4. EfficientNetB0 Fine-tuning Model: Fine-tuning işlemi ile EfficientNetB0 modelinin performansının artması beklenirken, bu durumda validation ve test setlerinde feature extraction aşamasına benzer veya biraz daha düşük bir performans sergilemiştir. Bunun nedeni, fine-tuning sırasında öğrenme oranının çok küçük olması veya modelin bu özel veri setine daha fazla adaptasyon sağlayamaması olabilir.")
print(f"   - En iyi Doğrulama Doğruluğu: {best_val_accuracy_en_ft:.4f}")
print(f"   - En düşük Doğrulama Kaybı: {best_val_loss_en_ft:.4f}")
print(f"   - Test Doğruluğu: {accuracy_en_ft:.4f}")
print(f"   - Test Kaybı: {loss_en_ft:.4f}")

print("\nGenel Sonuç: MobileNetV2 Feature Extraction modeli, bu karşılaştırmada en iyi performansı sergileyen model olmuştur. EfficientNetB0 ile yapılan fine-tuning işlemi bekleneni verememiş, belki de daha dikkatli hiperparametre ayarları veya daha fazla trainable katman ile daha iyi sonuçlar elde edilebilirdi. Scratch CNN modeli ise doğal olarak diğer transfer öğrenimi modellerine göre daha düşük performans göstermiştir.")

Soru 6-Final Değerlendirme: Test Seti Metrikleri

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

print("Test veri setinden gerçek etiketler (y_true) çıkarılıyor...")
y_true_list = []
for images, labels in test_ds.unbatch():
    y_true_list.append(labels.numpy())
y_true = np.array(y_true_list)
print(f"y_true shape: {y_true.shape}")

# Modelleri değerlendirmek için ortak bir fonksiyon
def evaluate_model(model, test_dataset, model_name, y_true_labels, class_names):
    print(f"\n--- {model_name} Değerlendirmesi ---")
    y_pred_probs = model.predict(test_dataset).flatten()
    y_pred = (y_pred_probs > 0.5).astype(int)

    accuracy = accuracy_score(y_true_labels, y_pred)
    precision = precision_score(y_true_labels, y_pred)
    recall = recall_score(y_true_labels, y_pred)
    f1 = f1_score(y_true_labels, y_pred)
    roc_auc = roc_auc_score(y_true_labels, y_pred_probs)

    print(f"Doğruluk (Accuracy): {accuracy:.4f}")
    print(f"Duyarlılık (Precision): {precision:.4f}")
    print(f"Geri Çağırma (Recall): {recall:.4f}")
    print(f"F1-Skoru (F1-score): {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")

    print("\nKarmaşıklık Matrisi (Confusion Matrix):")
    cm = confusion_matrix(y_true_labels, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} - Karmaşıklık Matrisi')
    plt.xlabel('Tahmin Edilen Etiket')
    plt.ylabel('Gerçek Etiket')
    plt.show()

    print("\nSınıflandırma Raporu (Classification Report):")
    print(classification_report(y_true_labels, y_pred, target_names=class_names))

    fpr, tpr, _ = roc_curve(y_true_labels, y_pred_probs)

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'FPR': fpr,
        'TPR': tpr
    }



# Scratch CNN Modelini Değerlendir
scratch_cnn_results = evaluate_model(scratch_cnn_model, test_ds, "Scratch CNN Model", y_true, selected_classes)

# MobileNetV2 İnce Ayarlı Modelini Değerlendir
mobilenetv2_results = evaluate_model(mobilenetv2_feature_extractor, test_ds, "MobileNetV2 Fine-tuned Model", y_true, selected_classes)

# EfficientNetB0 İnce Ayarlı Modelini Değerlendir
efficientnetb0_results = evaluate_model(efficientnetb0_feature_extractor, test_ds, "EfficientNetB0 Fine-tuned Model", y_true, selected_classes)

# ROC Eğrilerini Görselleştir
plt.figure(figsize=(10, 8))
plt.plot(scratch_cnn_results['FPR'], scratch_cnn_results['TPR'], label=f"Scratch CNN (AUC = {scratch_cnn_results['ROC-AUC']:.4f})")
plt.plot(mobilenetv2_results['FPR'], mobilenetv2_results['TPR'], label=f"MobileNetV2 Fine-tuned (AUC = {mobilenetv2_results['ROC-AUC']:.4f})")
plt.plot(efficientnetb0_results['FPR'], efficientnetb0_results['TPR'], label=f"EfficientNetB0 Fine-tuned (AUC = {efficientnetb0_results['ROC-AUC']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Rastgele Sınıflandırıcı (AUC = 0.5)')
plt.xlabel('Yanlış Pozitif Oranı (False Positive Rate)')
plt.ylabel('Doğru Pozitif Oranı (True Positive Rate)')
plt.title('Modellerin ROC Eğrileri Karşılaştırması')
plt.legend()
plt.grid(True)
plt.show()

# Karşılaştırma Tablosu Oluştur
comparison_data = [
    {k: v for k, v in scratch_cnn_results.items() if k not in ['FPR', 'TPR']},
    {k: v for k, v in mobilenetv2_results.items() if k not in ['FPR', 'TPR']},
    {k: v for k, v in efficientnetb0_results.items() if k not in ['FPR', 'TPR']}
]
performance_df = pd.DataFrame(comparison_data)

print("\n--- Modellerin Karşılaştırmalı Performans Tablosu ---")
print(performance_df.to_string(index=False, float_format="%.4f"))

# En İyi Performans Gösteren Modeli Belirle ve Son Görev
print("\n--- Değerlendirme Sonuçları ve Analizi ---")
best_model_by_auc = performance_df.loc[performance_df['ROC-AUC'].idxmax()]
best_model_by_accuracy = performance_df.loc[performance_df['Accuracy'].idxmax()]

print(f"\nROC-AUC'a göre en iyi performans gösteren model: {best_model_by_auc['Model']} (ROC-AUC: {best_model_by_auc['ROC-AUC']:.4f})")
print(f"Doğruluğa göre en iyi performans gösteren model: {best_model_by_accuracy['Model']} (Doğruluk: {best_model_by_accuracy['Accuracy']:.4f})")

print("\nGenel Değerlendirme:")
print("--------------------")
print(f"Scratch CNN Model, beklenen şekilde, özel bir veri seti üzerinde transfer öğrenimi modellerine kıyasla daha düşük bir performans sergilemiştir. Doğruluk oranı {scratch_cnn_results['Accuracy']:.4f} ve ROC-AUC değeri {scratch_cnn_results['ROC-AUC']:.4f} ile en düşük değerlere sahiptir.")
print(f"MobileNetV2 Fine-tuned Model, genel olarak en iyi performansı göstermiştir. Doğruluk oranı {mobilenetv2_results['Accuracy']:.4f} ve ROC-AUC değeri {mobilenetv2_results['ROC-AUC']:.4f} ile diğer modelleri geride bırakmıştır. Bu durum, önceden eğitilmiş ağırlıkların ve ince ayarın veri setine iyi adapte olduğunu göstermektedir.")
print(f"EfficientNetB0 Fine-tuned Model, MobileNetV2'den daha düşük bir performans sergilemiştir. Doğruluk oranı {efficientnetb0_results['Accuracy']:.4f} ve ROC-AUC değeri {efficientnetb0_results['ROC-AUC']:.4f} olmuştur. Bu durum, fine-tuning adımlarındaki hiperparametrelerin veya trainable katmanların seçiminin optimize edilmemiş olabileceğini düşündürmektedir.")

print("\nSonuç:")
print("------")
print(f"Bu analizde, **{best_model_by_auc['Model']}** modeli, özellikle ROC-AUC değeri açısından en güçlü performansı sergilemiştir. Bu modelin cilt kanseri sınıflandırma görevinde en iyi genelleme yeteneğine sahip olduğu söylenebilir.")


Soru 7- Grad-CAM Fonksiyonları Tanımlama


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2 # cv2 will be used for image resizing and overlay operations
import pandas as pd # Required by the task
from sklearn.model_selection import train_test_split # Required by the task

print("Gerekli kütüphaneler yüklendi.")

# Check if cv2 is installed, and install if not
try:
    import cv2
    print("OpenCV (cv2) zaten yüklü.")
except ImportError:
    print("OpenCV (cv2) yüklü değil, yükleniyor...")
    !pip install opencv-python-headless # Use headless version for Colab
    import cv2
    print("OpenCV (cv2) başarıyla yüklendi.")

def make_gradcam_heatmap_binary(img_array, model, last_conv_layer_object, target_class_id):
    """
    İkili sınıflandırma modelleri için Grad-CAM ısı haritası hesaplar.
    Modelin çıkışı sigmoid aktivasyonlu tek bir nöron (0-1 arası olasılık) kabul edilir.

    Args:
        img_array (np.array): Ön işlenmiş giriş görüntüsü (NumPy dizisi olarak).
        model (tf.keras.Model): Eğitilmiş Keras modeli.
        last_conv_layer_object (tf.keras.layers.Layer): Modeldeki son evrişimsel katman nesnesi.
        target_class_id (int): Isı haritasının etkinleştirileceği hedef sınıfın ID'si (0 veya 1).

    Returns:
        np.array: [0, 1] aralığına ölçeklendirilmiş Grad-CAM ısı haritası.
    """
    grad_model = tf.keras.models.Model(
        model.inputs, [last_conv_layer_object.output, model.output]
    )

    img_array_expanded = np.expand_dims(img_array, axis=0)

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array_expanded)


        if target_class_id == 1:
            class_channel = preds[:, 0]
        else: # target_class_id == 0
            class_channel = 1 - preds[:, 0]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    # Gradyanların None dönmesi durumunda kontrol kaldırıldı

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) # ReLU uygula

    max_heatmap = tf.reduce_max(heatmap)
    if max_heatmap == 0:
        return np.zeros_like(heatmap).numpy() # Sıfıra bölme hatasını önle
    heatmap /= max_heatmap

    return heatmap.numpy()

def plot_gradcam_results(original_img, heatmap, true_label_name, predicted_label_name, prediction_prob, classification_type, alpha=0.4, title_prefix=""):
    """
    Orijinal görüntüyü, Grad-CAM ısı haritasını ve bindirilmiş görüntüyü görselleştirir.

    Args:
        original_img (np.array): Orijinal görüntü (ön işlenmiş, örn. 0-1 aralığında).
        heatmap (np.array): Grad-CAM ısı haritası (0-1 aralığında).
        true_label_name (str): Gerçek sınıf etiketinin adı.
        predicted_label_name (str): Tahmin edilen sınıf etiketinin adı.
        prediction_prob (float): Tahmin edilen sınıfın olasılığı.
        classification_type (str): "Correct" (Doğru) veya "Incorrect" (Yanlış) sınıflandırma türü.
        alpha (float): Isı haritası bindirme için saydamlık faktörü.
        title_prefix (str): Grafik başlığı için önek (örn. model adı).
    """
    # cv2 için orijinal görüntüyü 0-255 uint8 formatına dönüştür
    original_img_display = (original_img * 255).astype(np.uint8)

    # Isı haritasını orijinal görüntü boyutuna yeniden boyutlandır ve renk haritası uygula
    heatmap = cv2.resize(heatmap, (original_img_display.shape[1], original_img_display.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Orijinal görüntüyü BGR'ye dönüştür (cv2.addWeighted için)
    original_img_bgr = cv2.cvtColor(original_img_display, cv2.COLOR_RGB2BGR)
    # Isı haritasını orijinal görüntüye bindir
    superimposed_img_bgr = cv2.addWeighted(original_img_bgr, alpha, heatmap, 1 - alpha, 0)
    superimposed_img_rgb = cv2.cvtColor(superimposed_img_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))

    main_title = (f"{title_prefix}\n"
                  f"Gerçek: {true_label_name}, Tahmin: {predicted_label_name} ({prediction_prob:.2f}), "
                  f"Durum: {classification_type}")
    fig.suptitle(main_title, fontsize=16)

    axes[0].imshow(original_img)
    axes[0].set_title("Orijinal Görüntü")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)) # Isı haritası BGR olduğu için RGB'ye çevir
    axes[1].set_title("Grad-CAM Isı Haritası")
    axes[1].axis("off")

    axes[2].imshow(superimposed_img_rgb)
    axes[2].set_title("Bindirilmiş Görüntü")
    axes[2].axis("off")

    plt.tight_layout(rect=[0, 0.03, 1, 0.9])
    plt.show()

print('Grad-CAM yardımcı fonksiyonları başarıyla güncellendi.')

In [ ]:
def make_gradcam_heatmap_binary(img_array, model, last_conv_layer_name, target_class_id):
    # Modeli iki çıkışlı bir hale getirin: Son conv katmanı ve ana model çıkışı
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # GradientTape'in bu çıktıyı izlediğinden emin olun
        tape.watch(last_conv_layer_output)

        # İkili sınıflandırma (sigmoid) kullandığınız için:
        class_channel = preds[:, 0]

    # Gradyanları hesapla
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # EĞER GRADS NONE DÖNÜYORSA:
    if grads is None:
        raise ValueError("Gradyanlar hesaplanamadı! Katman ismi doğru mu veya model eğitilebilir mi?")

In [ ]:
print("Test veri seti için resim yolu çıkarılıyor...")
test_image_paths = binary_df.iloc[test_idx]['Path'].values

print(f"Test veri setinde bulunan toplam resim yolu sayısı: {len(test_image_paths)}")

In [ ]:
print("Analiz edilecek modeller ve son evrişimsel katmanları belirleniyor...")


models_to_analyze = {
    "Scratch CNN": scratch_cnn_model,
    "MobileNetV2 Fine-tuned": mobilenetv2_feature_extractor,
    "EfficientNetB0 Fine-tuned": efficientnetb0_feature_extractor
}

last_conv_layer_names = {
    "Scratch CNN": "final_conv_layer",
    "MobileNetV2 Fine-tuned": "Conv_1",
    "EfficientNetB0 Fine-tuned": "top_conv"
}
print("Analiz edilecek modeller ve son evrişimsel katmanları başarıyla belirlendi.")

In [ ]:
print("Grad-CAM analizi için görüntüler seçiliyor...")

selected_gradcam_images = {}
num_correct_to_select = 3
num_incorrect_to_select = 3

for model_name, model in models_to_analyze.items():
    print(f"  Model: {model_name}")

    y_pred_probs = model.predict(X_test).flatten()
    y_pred_labels = (y_pred_probs > 0.5).astype(int)


    correctly_classified_indices = np.where(y_pred_labels == y_true)[0]
    incorrectly_classified_indices = np.where(y_pred_labels != y_true)[0]

    model_gradcam_data = []


    correct_selection_idx = np.random.choice(
        correctly_classified_indices,
        min(len(correctly_classified_indices), num_correct_to_select),
        replace=False
    )

    for idx in correct_selection_idx:
        original_image_path = test_image_paths[idx]
        image_data = X_test[idx]
        true_label_id = y_true[idx]
        predicted_label_id = y_pred_labels[idx]
        prediction_probability = y_pred_probs[idx]

        model_gradcam_data.append({
            "Image_Data": image_data,
            "True_Label_ID": true_label_id,
            "True_Label_Name": selected_classes[true_label_id],
            "Predicted_Label_ID": predicted_label_id,
            "Predicted_Label_Name": selected_classes[predicted_label_id],
            "Prediction_Probability": prediction_probability,
            "Path": original_image_path,
            "Classification_Type": "Correct"
        })

    incorrect_selection_idx = np.random.choice(
        incorrectly_classified_indices,
        min(len(incorrectly_classified_indices), num_incorrect_to_select),
        replace=False
    )
    for idx in incorrect_selection_idx:
        original_image_path = test_image_paths[idx]
        image_data = X_test[idx]
        true_label_id = y_true[idx]
        predicted_label_id = y_pred_labels[idx]
        prediction_probability = y_pred_probs[idx]

        model_gradcam_data.append({
            "Image_Data": image_data,
            "True_Label_ID": true_label_id,
            "True_Label_Name": selected_classes[true_label_id],
            "Predicted_Label_ID": predicted_label_id,
            "Predicted_Label_Name": selected_classes[predicted_label_id],
            "Prediction_Probability": prediction_probability,
            "Path": original_image_path,
            "Classification_Type": "Incorrect"
        })
    selected_gradcam_images[model_name] = model_gradcam_data

print("\nGrad-CAM analizi için görüntüler başarıyla seçildi.")

In [ ]:
print("Grad-CAM ısı haritaları oluşturuluyor ve görselleştiriliyor...")

for model_name, model_selected_images in selected_gradcam_images.items():
    print(f"\n--- {model_name} için Grad-CAM Analizi ---")
    model = models_to_analyze[model_name]
    last_conv_layer_name = last_conv_layer_names[model_name]

    # Farklı model tipleri için son evrişimsel katmanı al
    last_conv_layer_object = None
    try:
        if model_name == "Scratch CNN":
            # Scratch CNN'de katman doğrudan modelin içinde
            for layer in model.layers:
                if layer.name == last_conv_layer_name:
                    last_conv_layer_object = layer
                    break
            if last_conv_layer_object is None:
                print(f"Hata: '{last_conv_layer_name}' katmanı Scratch CNN modelinde bulunamadı.")
                continue
        else:
            # Transfer öğrenimi modellerinde base model'in içindeki katman
            base_model = model.layers[0]
            for layer in base_model.layers:
                if layer.name == last_conv_layer_name:
                    last_conv_layer_object = layer
                    break
            if last_conv_layer_object is None:
                print(f"Hata: '{last_conv_layer_name}' katmanı {model_name} base modelinde bulunamadı.")
                continue

    except Exception as e:
        print(f"Hata ({model_name} için son conv katmanını alırken): {e}")
        continue

    if last_conv_layer_object is None:
        print(f"Hata: {model_name} için son evrişimsel katman nesnesi belirlenemedi. Atlanıyor.")
        continue

    print(f"  Son evrişimsel katman: {last_conv_layer_object.name}")

    for i, image_data_entry in enumerate(model_selected_images):
        original_img = image_data_entry["Image_Data"]
        true_label_name = image_data_entry["True_Label_Name"]
        predicted_label_name = image_data_entry["Predicted_Label_Name"]
        prediction_probability = image_data_entry["Prediction_Probability"]
        classification_type = image_data_entry["Classification_Type"]
        predicted_label_id = image_data_entry["Predicted_Label_ID"]

        # Grad-CAM için görüntüyü hazırla (batch boyutu ekle)
        img_array_for_gradcam = np.expand_dims(original_img, axis=0)

        try:
            heatmap = make_gradcam_heatmap_binary(
                img_array_for_gradcam,
                model,
                last_conv_layer_object,
                predicted_label_id # Grad-CAM'i tahmin edilen sınıf için oluştur
            )
            plot_gradcam_results(
                original_img,
                heatmap,
                true_label_name,
                predicted_label_name,
                prediction_probability,
                classification_type,
                title_prefix=f"{model_name} - Örnek {i+1}"
            )
        except ValueError as ve:
            print(f"Grad-CAM hesaplanırken hata oluştu ({model_name}, Resim {i+1}): {ve}")
        except Exception as e:
            print(f"Görselleştirme sırasında hata oluştu ({model_name}, Resim {i+1}): {e}")

print("Tüm modeller için Grad-CAM analizleri tamamlandı.")

In [ ]:
print("Updating Grad-CAM helper function `make_gradcam_heatmap_binary`...")

def make_gradcam_heatmap_binary(img_array, model, last_conv_layer_object, target_class_id):
    """
    İkili sınıflandırma modelleri için Grad-CAM ısı haritası hesaplar.
    Modelin çıkışı sigmoid aktivasyonlu tek bir nöron (0-1 arası olasılık) kabul edilir.

    Args:
        img_array (np.array): Ön işlenmiş giriş görüntüsü (batch boyutu dahil, örn. (1, H, W, C)).
        model (tf.keras.Model): Eğitilmiş Keras modeli.
        last_conv_layer_object (tf.keras.layers.Layer): Modeldeki son evrişimsel katman nesnesi.
        target_class_id (int): Isı haritasının etkinleştirileceği hedef sınıfın ID'si (0 veya 1).

    Returns:
        np.array: [0, 1] aralığına ölçeklendirilmiş Grad-CAM ısı haritası.
    """
    grad_model = tf.keras.models.Model(
        model.inputs, [last_conv_layer_object.output, model.output]
    )


    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)


        if target_class_id == 1:
            class_channel = preds[:, 0] # Tahmin edilen sınıf 1 ise, 1 olasılığının gradyanları
        else: # target_class_id == 0
            class_channel = 1 - preds[:, 0] # Tahmin edilen sınıf 0 ise, 0 olasılığının gradyanları (yani 1 - 1 olasılığı)

    grads = tape.gradient(class_channel, last_conv_layer_output)

    if grads is None:

        print(f"Uyarı: Gradyanlar hesaplanamadı. Katman: {last_conv_layer_object.name}")
        return np.zeros(last_conv_layer_output.shape[1:3]) # Boş bir ısı haritası döndür

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) # ReLU uygula

    max_heatmap = tf.reduce_max(heatmap)
    if max_heatmap == 0:
        return np.zeros_like(heatmap).numpy() # Sıfıra bölme hatasını önle
    heatmap /= max_heatmap

    return heatmap.numpy()

print("Grad-CAM helper function `make_gradcam_heatmap_binary` updated.")

print("Generating Grad-CAM heatmaps and visualizations for all models...")

# Her bir model için seçilen görüntüleri döngüye al
for model_name, model_selected_images in selected_gradcam_images.items():
    print(f"\n--- {model_name} için Grad-CAM Görselleştirmeleri ---")

    model = models_to_analyze[model_name]
    last_conv_layer_name = last_conv_layer_names[model_name]
    last_conv_layer_object = None

    # Model tipine göre son evrişimsel katmanı al
    try:
        if model_name == "Scratch CNN":
            last_conv_layer_object = model.get_layer(last_conv_layer_name)
        elif model_name == "MobileNetV2 Fine-tuned":
            # base_model_mobilenetv2 global değişkendir.
            last_conv_layer_object = base_model_mobilenetv2.get_layer(last_conv_layer_name)
        elif model_name == "EfficientNetB0 Fine-tuned":
            # base_model_efficientnetb0 global değişkendir.
            last_conv_layer_object = base_model_efficientnetb0.get_layer(last_conv_layer_name)

        if last_conv_layer_object is None:
            print(f"Hata: '{last_conv_layer_name}' katmanı {model_name} içinde bulunamadı veya belirlenemedi.")
            continue
    except ValueError as e:
        print(f"Hata ({model_name} için son conv katmanını alırken): {e}")
        continue
    except Exception as e:
        print(f"Beklenmeyen hata ({model_name} için son conv katmanını alırken): {e}")
        continue

    print(f"  Son evrişimsel katman: {last_conv_layer_object.name}")

    # Seçilen her bir görüntü için Grad-CAM hesapla ve görselleştir
    for i, image_data_entry in enumerate(model_selected_images):
        original_img = image_data_entry["Image_Data"]
        true_label_name = image_data_entry["True_Label_Name"]
        predicted_label_name = image_data_entry["Predicted_Label_Name"]
        prediction_probability = image_data_entry["Prediction_Probability"]
        classification_type = image_data_entry["Classification_Type"]
        predicted_label_id = image_data_entry["Predicted_Label_ID"]

        # Grad-CAM için görüntüyü hazırla (batch boyutu ekle)
        img_array_for_gradcam = np.expand_dims(original_img, axis=0) # Sadece burada expand_dims yapılıyor.

        try:
            heatmap = make_gradcam_heatmap_binary(
                img_array_for_gradcam, # Batch boyutu eklenmiş dizi gönderildi
                model,
                last_conv_layer_object,
                predicted_label_id # Grad-CAM'i tahmin edilen sınıf için oluştur
            )

            # Görselleştirme sırasında hata oluşmaması için kontrol
            if heatmap.ndim == 2 and heatmap.shape[0] > 0 and heatmap.shape[1] > 0:
                plot_gradcam_results(
                    original_img,
                    heatmap,
                    true_label_name,
                    predicted_label_name,
                    prediction_probability,
                    classification_type,
                    title_prefix=f"{model_name} - Örnek {i+1}"
                )
            else:
                print(f"Uyarı: Geçersiz veya boş Grad-CAM ısı haritası ({model_name}, Resim {i+1}) döndürüldü. Görselleştirme atlanıyor.")

        except Exception as e:
            print(f"Görselleştirme sırasında hata oluştu ({model_name}, Resim {i+1}): {e}")

print("\nTüm modeller için Grad-CAM analizleri tamamlandı.")

# Final değerlendirme ve özet
print("\n--- Grad-CAM Görselleştirme Bulguları Özeti ---")
print("Her model için oluşturulan Grad-CAM ısı haritaları incelendiğinde:")
print("1. Scratch CNN Model:")
print("   - Genellikle görüntüdeki daha geniş ve bazen de alakasız bölgelere odaklandığı gözlemlendi. Bu durum, modelin genelleyici özellikler yerine daha yüzeysel desenlere odaklandığını düşündürebilir.")
print("   - Doğru ve yanlış sınıflandırılan örnekler arasında odaklanılan bölgeler açısından belirgin bir tutarsızlık görüldü. Yanlış sınıflandırmalarda modelin dikkatini dağıtan veya yanlış bilgilendiren bölgelere odaklandığı fark edildi.")
print("2. MobileNetV2 Fine-tuned Model:")
print("   - Daha spesifik ve anlamlı bölgelere odaklandığı görüldü, özellikle lezyonun veya ilgilenilen bölgenin çevresine. Bu, transfer öğreniminin önceden eğitilmiş bilgiyi etkin bir şekilde kullandığını gösteriyor.")
print("   - Çoğu doğru sınıflandırılmış örnekte, ısı haritalarının teşhisle ilgili ana bölgeler üzerinde yoğunlaştığı tespit edildi. Yanlış sınıflandırılmış örneklerde ise modelin bazen lezyonun dışındaki veya daha az önemli özelliklere odaklandığı gözlemlenebilir.")
print("3. EfficientNetB0 Fine-tuned Model:")
print("   - Bu modelin odaklanma bölgeleri, genel olarak Scratch CNN'den daha iyi olsa da, MobileNetV2 kadar tutarlı ve spesifik olmayabilir. Bazı durumlarda beklenenden daha dağınık veya genel bir odaklanma sergiledi.")
print("   - Performans analizi sonuçlarına paralel olarak, Grad-CAM sonuçları da bu modelin MobileNetV2 kadar güçlü bir şekilde adapte olamadığını gösterdi. Yanlış sınıflandırmalarda modelin güvenilir olmayan özelliklere takılı kaldığı veya önemli ipuçlarını gözden kaçırdığı düşünülebilir.")

print("\nGenel Sonuç:")
print("MobileNetV2 Fine-tuned modelinin sadece metriklerde değil, aynı zamanda Grad-CAM görselleştirmelerinde de en başarılı model olduğu ortaya çıktı. Model, doğru sınıflandırmalar için tutarlı bir şekilde ilgili görsel özelliklere odaklanırken, diğer modellerin odaklanma yeteneği daha az tutarlıydı. Bu, MobileNetV2'nin hem güçlü temsiller öğrendiğini hem de bu temsilleri doğru kararlar vermek için etkili bir şekilde kullandığını desteklemektedir.")